# 지시를 따르도록 미세 튜닝하기 목차
* [Chapter 1 지시 미세 튜닝 소개](#chapter1)
* [Chapter 2 지도 학습 지시 미세 튜닝을 위해 데이터셋 준비하기](#chapter2)
* [Chapter 3 훈련 배치 만들기](#chapter3)

## Chapter 1 지시 미세 튜닝 소개 <a class="anchor" id="chapter1"></a>
1. LLM을 사전 훈련하는 것은 한번에 한 단어씩 생성하는 방법을 배우넌 것입니다.
   - 이렇게 만들어진 사전 훈련된 LLM은 텍스트 완성능력이 있다.
   - "이 텍스트 문법을 고쳐 줘"와 같은 구체적인 명령을 잘 수행하지 못합니다.

2. 지시를 따르고 기대하는 응답을 생성하도록 LLM 능력을 향상시켜봅시다.

    ![예시](image/07-01-example5.png)

3. 지시 미세튜닝은 세 단계 과정을 거친다.

   ![순서](image/07-01-process3.png)

## Chapter 2 지도 학습 지시 미세 튜닝을 위해 데이터셋 준비하기 <a class="anchor" id="chapter2"></a>
1. 지시 미세 튜닝을 위해서는 지시와 그에 대한 응답이 포함된 데이터셋이 필요하다.
   - 예: "이 텍스트 문법을 고쳐 줘" -> "고쳐진 텍스트"

In [4]:
# 데이터셋 다운로드
import json
import os
import urllib

def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode('utf-8')
        with open(file_path,'w', encoding='utf-8') as file:
            file.write(text_data)
    with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)
    return data

file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
print("샘플 개수:", len(data))


샘플 개수: 1100


2. 샘플 데이터 양식은 'instruction', 'input', 'output' 키로 구성된 파이썬 딕셔너리의 리스트이다.
    - 'instruction': 모델이 수행해야 할 작업 지시
    - 'input': 작업에 필요한 추가 정보(없을 수도 있음)
    - 'output': 모델이 생성해야 할 기대 응답

In [ ]:
# 샘플 데이터 출력
#   - 'instruction', 'input', 'output' 키로 구성된 파이썬 딕셔너리의 리스트
print("샘플 데이터 예시:\n", data[50])

# input 필드가 비어있는 경우도 있음
#   - 
print("다른 샘플 데이터 예시:\n", data[999])

샘플 데이터 예시:
 {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}
다른 샘플 데이터 예시:
 {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."}



3. input 필드가 비어있는 경우도 있음
   - 예: "이 텍스트 문법을 고쳐 줘" -> "고쳐진 텍스트"

4. LLM을 위해 샘플을 포멧팅하는 방법은 여러가지가 있다.
   - 알파카(Alpaca) 스타일 포멧
      - 지시, 입력, 응답 섹션으로 구성된다.
   - Phi-3 스타일 포멧
      - <|user|>와 <|assistant|> 토큰으로 구성된 간단한 포멧을 사용한다.
   - 이른 종종 프롬프트 스타일이라고 부른다.

      ![스타일](image/07-01-style2.png)

5. 알파카는 초기 LLM 중 하나로 지시 미세 튜닝 과정에 대한 내용이 공개되어 있다.
   - 알파카 스타일 포멧이 널리 사용된다.
   - 본 노트북에서는 알파카 스타일 포멧을 사용한다.

In [9]:
# data 리스트의 항목을 알파카 스타일 포맷으로 변환
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

In [10]:
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"
print(model_input+desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

### Response:
The correct spelling is 'Occasion.'


In [11]:
# format_input 함수는 값이 비어 있을 때 input 섹션을 건너뜁니다.
model_input_no_input = format_input(data[999])
desired_response_no_input = f"\n\n### Response:\n{data[999]['output']}"
print(model_input_no_input+desired_response_no_input)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response:
An antonym of 'complicated' is 'simple'.


6. 데이터 셋츨 훈련 세트, 검증 세트, 테스트 세트로 나눈다.

In [13]:
train_portion = int(len(data) * 0.85)
val_portion = int(len(data) * 0.1)
test_portion = len(data) - train_portion - val_portion

train_data = data[:train_portion]
val_data = data[train_portion:train_portion + val_portion]
test_data = data[train_portion + val_portion:]

print("훈련 세트 크기:", len(train_data))
print("테스트 세트 크기:", len(test_data))
print("검증 세트 크기:", len(val_data))

훈련 세트 크기: 935
테스트 세트 크기: 55
검증 세트 크기: 110


## Chapter 3 훈련 배치 만들기 <a class="anchor" id="chapter3"></a>
1. 지시 미세 튜닝을 위해 훈련 배치를 만드는 방법을 알아봅시다.

    ![순서](image/07-03-process.png)

2. 이전 장에서 파이토치 DataLoader 클래스로 훈련 배치를 자동으로 생성하였다.
   - 샘플 리스트를 배치로 묶어주는 기본 콜레이드(collate) 함수를 사용하였다.
   - 콜레이트 함수는 훈련하는 동안 개별 데이터 샘플의 리스트를 받아 하나의 배치로 합쳐 모델이 효과적으로 처리할 수 있도록 한다.
   - 지시 미세 튜닝을 위한 배치 구성은 조금 더 복잡하기 때문에 DataLoader에 적용할 사용자 정의 콜레이트 함수를 만들어야 한다.

      ![배치](image/07-03-batch2.png)

3. 배치 과정을 구현하기 위해 처음 두 단계를 진행
   - 특정 프롬프트 템플릿으로 샘플을 포맷팅
   - 토크나이저를 사용해 텍스트를 토큰으로 변환한다.

      ![단계](image/07-03-step.png) 



In [ ]:
import torch
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_text = []
        for entry in data: # 텍스트를 토큰화 한다.
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_text.append(
                tokenizer.encode(full_text)
            )
            
    def __getitem__(self, index):
        return self.encoded_text[index]
    
    def __len__(self):
        return len(self.data)

4. 여러개의 훈련 샘플을 배치로 묶어 훈련 속도를 높인다.
   - 모든 입력의 길이가 같도록 패딩한다.
   - |endoftext| 토큰의 토큰ID를 패딩 토큰ID로 사용한다.

In [3]:
# |endoftext| 토큰의 토큰ID 확인
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
eot_token_id = tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})
print("패딩 토큰 ID:", eot_token_id)

패딩 토큰 ID: [50256]


5. 조금 더 복잡한 작업을 수행하기 위해 데이터 로드데 전달할 사용자 정의 콜레이드 함수를 작성한다.
   - 배치에 있는 훈련 샘플의 길이가 동일하도록 패딩을 추가한다.
   - 각 배치에서 가장 긴 샘플의 길이에 맞춘다.
   - 첫 번째 배치와 두 번째 배치에서 각 배치의 길이가 다를 수 있다

      ![단계2](image/07-03-step2.png) 

In [14]:
import torch

def custom_collate_draft_1(batch, pad_token_id=50256, device='cpu'):
    # 1. 배치에서 가장 긴 샘플의 길이를 찾는다.
    batch_max_length = max(len(item)+1 for item in batch)
    
    # 2. 각 샘플을 패딩하여 동일한 길이로 만든다.
    inputs_lst = []
    for item in batch:
        new_item = item.copy()
        # 2-1. 샘플 끝에 패딩 토큰을 추가한다
        new_item+= [ pad_token_id]
        
        # 2-2. 샘플 길이가 배치 최대 길이에 도달할 때까지 패딩 토큰을 추가한다
        padded = (new_item + 
                  [pad_token_id] * (batch_max_length - len(new_item)))
        
        inputs = torch.tensor(padded[:-1]) # 마지막 토큰은 제외
        inputs_lst.append(inputs)
    
    input_tensor = torch.stack(inputs_lst).to(device)

    return input_tensor

In [24]:
# custom_collate_draft_1 함수 테스트
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = (
    inputs_1,
    inputs_2,
    inputs_3
)

print(custom_collate_draft_1(batch))

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])


6. 입력 토큰 ID에 대한 타킷 토큰 ID로 구성된 배치 생성
   - 사용자 정의 콜레이트 함수를 입력 토큰 ID와 타깃 토큰 ID를 반환하도록 수정한다.
   - 타깃 시퀸스는 입력 시퀸스의 토큰 ID를 하나씩 오른쪽으로 이동하여 생성한다.

     ![패딩](image/07-03-padding.png) 

In [25]:
def custom_collate_draft_2(batch, pad_token_id=50256, device='cpu'):
    # 1. 배치에서 가장 긴 샘플의 길이를 찾는다.
    batch_max_length = max(len(item)+1 for item in batch)
    
    # 2. 각 샘플을 패딩하여 동일한 길이로 만든다.
    inputs_lst = []
    targets_lst = []
    for item in batch:
        new_item = item.copy()
        # 2-1. 샘플 끝에 패딩 토큰을 추가한다
        new_item+= [ pad_token_id]
        
        # 2-2. 샘플 길이가 배치 최대 길이에 도달할 때까지 패딩 토큰을 추가한다
        padded = (new_item + 
                  [pad_token_id] * (batch_max_length - len(new_item)))
        
        inputs = torch.tensor(padded[:-1]) # 마지막 토큰은 제외
        targets = torch.tensor(padded[1:]) # 첫 번째 토큰은 제외
        
        inputs_lst.append(inputs)
        targets_lst.append(targets)
    
    input_tensor = torch.stack(inputs_lst).to(device)
    target_tensor = torch.stack(targets_lst).to(device)

    return input_tensor, target_tensor

In [27]:
# custom_collate_draft_2 함수 테스트
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = (
    inputs_1,
    inputs_2,
    inputs_3
)
print(custom_collate_draft_2(batch))

(tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]]), tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256, 50256, 50256, 50256],
        [    8,     9, 50256, 50256, 50256]]))


7. 모든 패딩 토큰을 플레이스 홀더 값 -100으로 대체

8. 토큰 아이디 가 50256인 종료 토큰을 하나 남겨둔다.
   - LLM이 지시에 응답할 때 텍스트 종료 토큰을 언제 생성할 지 학습할 수 있다.

      ![종료](image/07-03-end.png) 

9. 사용자 콜레이트 함수를 수정하여 타깃 리스트에서 아이디가 50256인 토큰을 -100으로 변경한다.


In [39]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    # 배치에서 가장 긴 시퀀스 찾기
    batch_max_length = max(len(item)+1 for item in batch)

    # 입력과 타깃 패딩 및 준비
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        # <|endoftext|> 토큰 추가
        new_item += [pad_token_id]
        # 시퀀스를 max_length까지 패딩
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])  # 입력을 위해 마지막 토큰 자르기
        targets = torch.tensor(padded[1:])  # 목표를 위해 오른쪽으로 +1 이동

        # 새로 추가: 목표에서 첫 번째 패딩 토큰을 제외한 모든 토큰을 ignore_index로 바꾸기
        mask = targets == pad_token_id # 50256 토큰아이디를 가진 타겟의 위치
        indices = torch.nonzero(mask).squeeze()
        # numel(): 텐서의 원소 개수를 반환
        if indices.numel() > 1: # 첫 번째 패딩 토큰 이후의 모든 패딩 토큰 위치
            targets[indices[1:]] = ignore_index
        
        # 새로 추가: 최대 시퀀스 길이로 자르기 (선택 사항)
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # 입력 및 타깃 리스트를 텐서로 변환하고 타깃 장치로 전송
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

In [40]:
inputs, targets = custom_collate_fn(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])


10. 타깃 토큰에서 -100 플레이스 홀더 사용 시 손실 계산에서 제외된다.
    - 파이토치의 크로스엔트로피 함수의 기본 설정은 ignore_index=-100 이다.
    - LLM이 응답의 끝을 나타내는 텍스트 종료 토큰을 생성하는 방법을 할습 할 수 있을독 타깃에서 하나의 50256은 남겨둔다.

11. 지시에 해당하는 타깃 토큰 ID를 마스킹하는 것이 일반적이다.
   - 생성된 응답 타깃 ID에서만 크로스 엔트로피 손실을 계산한다.

In [42]:
logits_1 = torch.tensor([
                            [-1.0, 1.0], # 첫 번째 토큰에 대한 예측
                            [-0.5, 1.5]  # 두 번째 토큰에 대한 예측
                        ])
targets_1 = torch.tensor([0, 1]) # 정답 토큰 인덱스
loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
print("손실값 1:", loss_1)   

손실값 1: tensor(1.1269)


In [43]:
# 토큰 ID를 추가하여 손실에 영향을 준다
logits_2 = torch.tensor([
                            [-1.0, 1.0], # 첫 번째 토큰에 대한 예측
                            [-0.5, 1.5],  # 두 번째 토큰에 대한 예측
                            [-0.5, 1.5] # 세 번째 토큰에 대한 예측
                        ])
targets_2 = torch.tensor([0, 1, 1]) # 정답 토큰 인덱스
loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print("손실값 2:", loss_2)  

손실값 2: tensor(0.7936)


In [44]:
# 마지막 토큰 아이디를 -100으로 변경
targets_3 = torch.tensor([0, 1, -100]) # 정답 토큰 인덱스
loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
print("손실값 3:", loss_3)

손실값 3: tensor(1.1269)


12. 파이토치의 크로스엔트로피 함수의 기본 설정은 ignore_index=-100 이다.
   - 이 설정은 손실 계산에서 -100 인덱스를 가진 토큰을 무시한다.

13. 지시에 해당하는 타깃 토큰 ID를 마스킹하는 것이 유용한지에 대해 연구자들의 의견이 갈린다.
   - 생성된 응답 타깃 ID에서만 크로스 엔트로피 손실을 계산한다.

        ![마스킹](image/07-03-masking.png) 